# Filters in C → Arduino

Write your filter in C here. We compile it with `gcc`, plot the result, then you paste the function straight into your Arduino sketch.

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from rocket_signal import generate
%matplotlib inline

t, true_sig = generate()
noisy = true_sig + 2.0 * np.random.default_rng(0).standard_normal(t.size)
np.savetxt('input.csv', noisy, fmt='%.6f')   # the C program reads this

## SMA

$$y_n = \frac{1}{N}(x_n + x_{n-1} + \dots + x_{n-N+1})$$

Average the last $N$ samples. Fill in the `TODO`.

In [ ]:
%%writefile sma.c
#include <stdio.h>
#define N 20

float sma_update(float x) {
    static float buf[N], sum = 0;
    static int idx = 0, count = 0;

    // TODO: drop oldest, store new, update sum, advance idx, return average.

    return 0.0f;
}

int main(void) {
    FILE *f = fopen("input.csv","r"); float x; int n=0;
    while (fscanf(f,"%f",&x)==1) printf("%d,%.6f\n", n++, sma_update(x));
}

In [ ]:
!gcc -O2 -o sma sma.c && ./sma > sma_out.csv
y = np.loadtxt('sma_out.csv', delimiter=',')[:,1]
plt.figure(figsize=(10,4))
plt.plot(t, noisy, color='lightgray', lw=0.8, label='Noisy')
plt.plot(t, true_sig, 'k--', lw=1.2, label='True')
plt.plot(t, y, 'royalblue', lw=2, label='Your SMA')
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

<details><summary>Stuck? Reference</summary>

```c
sum -= buf[idx];
buf[idx] = x;
sum += x;
idx = (idx + 1) % N;
if (count < N) count++;
return sum / count;
```
</details>

**Arduino:** paste `sma_update` as-is. Call it from `loop()`:

```cpp
float smoothed = sma_update(analogRead(A0));
Serial.println(smoothed);
delay(20);
```

## EMA

$$y_n = \alpha\, x_n + (1 - \alpha)\, y_{n-1}$$

One number of state. First sample: just return `x`.

In [ ]:
%%writefile ema.c
#include <stdio.h>
#define ALPHA 0.10f

float ema_update(float x) {
    static float y = 0;
    static int first = 1;

    // TODO: if first sample, set y = x and return. Otherwise apply the formula.

    return 0.0f;
}

int main(void) {
    FILE *f = fopen("input.csv","r"); float x; int n=0;
    while (fscanf(f,"%f",&x)==1) printf("%d,%.6f\n", n++, ema_update(x));
}

In [ ]:
!gcc -O2 -o ema ema.c && ./ema > ema_out.csv
y = np.loadtxt('ema_out.csv', delimiter=',')[:,1]
plt.figure(figsize=(10,4))
plt.plot(t, noisy, color='lightgray', lw=0.8, label='Noisy')
plt.plot(t, true_sig, 'k--', lw=1.2, label='True')
plt.plot(t, y, 'crimson', lw=2, label='Your EMA')
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

<details><summary>Stuck? Reference</summary>

```c
if (first) { y = x; first = 0; return y; }
y = ALPHA * x + (1.0f - ALPHA) * y;
return y;
```
</details>

**Arduino:** paste `ema_update` as-is, call from `loop()` like the SMA above.